In [1]:
import os
from dotenv import load_dotenv
from groq import Groq
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import json

load_dotenv()
api_key = os.getenv("GROQ_API_KEY")

if not api_key:
    print("❌ GROQ_API_KEY not found. Please check your .env file.")
else:
    client = Groq(api_key=api_key)
    print("✅ Environment ready — API key loaded successfully.")

✅ Environment ready — API key loaded successfully.


In [11]:
def build_prompt(question, model_answer, student_answer):
    return f"""
You are a strict but fair examiner evaluating a student's written answer.

Judge based on: factual accuracy, completeness, and understanding (not just keyword matching).
Do NOT penalize different wording if the meaning is correct.
Do NOT give credit for correct-sounding phrases that show no real understanding.

QUESTION:
{question}

MODEL ANSWER:
{model_answer}

STUDENT ANSWER:
{student_answer}

Return your evaluation in EXACTLY this format, nothing else:

Score: X/10
Remarks:
- (short bullet point 1)
- (short bullet point 2)
- (short bullet point 3, only if applicable)
- (short bullet point 4, only if applicable)

Write each remark in direct instructor style — as if a teacher is marking the answer directly.
Do NOT start remarks with phrases like "The student..." or "The student's answer...".
Do NOT use soft analytical phrases like "is generally maintained" or "could be more clearly explained".
Keep each remark short and direct, like a real margin note on a graded paper —
straight to the point, not a full analytical sentence.

Example of WRONG style: "Factual accuracy is generally maintained, but lacks detail on valves."
Example of RIGHT style: "Missing valve details (tricuspid, mitral) in blood flow description."
"""


def get_evaluation(prompt):
    """Sends the prompt to the LLM. Returns None if the API call fails for any reason."""
    try:
        response = client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[{"role": "user", "content": prompt}]
        )
        return response.choices[0].message.content
    except Exception as e:
        print(f"⚠️ API call failed: {e}")
        return None


def parse_response(response_text):
    """Extracts score (int) and remarks (list) from the LLM's raw text.
    Returns (None, []) if the expected format isn't found, instead of crashing."""
    if not response_text:
        return None, []

    lines = response_text.strip().split("\n")
    score, remarks = None, []

    for line in lines:
        line = line.strip()
        if line.lower().startswith("score:"):
            try:
                score = int(line.split(":")[1].strip().split("/")[0].strip())
            except (ValueError, IndexError):
                score = None
        elif line.startswith("-"):
            remarks.append(line.lstrip("-").strip())

    return score, remarks


def evaluate_answer(question, model_answer, student_answer):
    """Master function: builds the prompt, calls the LLM, parses the result.
    Always returns a dictionary with 'score', 'remarks', and 'error' keys,
    so the calling code never has to guess what came back."""
    prompt = build_prompt(question, model_answer, student_answer)
    raw_response = get_evaluation(prompt)

    if raw_response is None:
        return {"score": None, "remarks": [], "error": "Could not reach the AI service. Please check your internet connection or API key and try again."}

    score, remarks = parse_response(raw_response)

    if score is None:
        return {"score": None, "remarks": [], "error": "The AI's response could not be understood. Please try again."}

    return {"score": score, "remarks": remarks, "error": None}

def save_report(result, filename="report.json"):
    """Saves question, student answer, score, and remarks to a JSON file."""
    report_data = {
        "score": result["score"],
        "remarks": result["remarks"]
    }
    with open(filename, "w", encoding="utf-8") as f:
        json.dump(report_data, f, indent=4, ensure_ascii=False)

In [12]:
sample_question = "Explain why the heart is considered a double pump, and describe the path blood takes through its four chambers."

sample_model_answer = """
The heart is called a double pump because it has two separate pumping systems working together in one organ.
The right side pumps deoxygenated blood to the lungs (pulmonary circulation), while the left side pumps
oxygenated blood to the rest of the body (systemic circulation).

Blood flow path: Deoxygenated blood enters the right atrium from the body via the vena cava. It passes through
the tricuspid valve into the right ventricle, which pumps it through the pulmonary artery to the lungs for
oxygenation. Oxygenated blood returns via the pulmonary vein into the left atrium, passes through the mitral
valve into the left ventricle, and is then pumped through the aorta to the rest of the body.

This dual-pump system ensures that oxygen-poor and oxygen-rich blood never mix, maintaining efficient oxygen
delivery throughout the body.
"""

sample_student_answer = """
The heart has 4 chambers, two atriums and two ventricles. Blood comes into the right atrium then goes to right
ventricle then to lungs to get oxygen. Then it comes back to left atrium then left ventricle then goes to whole
body. It is called double pump because it pumps blood two times, one time to lungs and one time to body.
"""

demo_result = evaluate_answer(sample_question, sample_model_answer, sample_student_answer)

if demo_result["error"]:
    print("⚠️", demo_result["error"])
else:
    print(f"Score: {demo_result['score']}/10\n")
    print("Remarks:")
    for r in demo_result["remarks"]:
        print(" -", r)

Score: 6/10

Remarks:
 - Incomplete explanation of double pump concept.
 - Missing valve details (tricuspid, mitral) in blood flow description.
 - Lacks mention of pulmonary artery and aorta in blood flow path.
 - Overly simplistic definition of double pump, lacks clarity on separate pumping systems.


In [13]:
# ---- Header ----
header = widgets.HTML("""
<div style="background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
            padding: 25px; border-radius: 12px; margin-bottom: 20px; text-align:center;">
    <h2 style="color:white; margin:0;">📚 AI Answer Evaluator</h2>
    <p style="color:#e0e0f0; margin:5px 0 0 0;">Compare a student's answer against a model answer using AI</p>
</div>
""")

# ---- Input fields (pre-filled with the documented sample) ----
box_layout = widgets.Layout(width="100%", height="90px", margin="0 0 12px 0")

question_box = widgets.Textarea(value=sample_question, layout=widgets.Layout(width="100%", height="50px"))
model_answer_box = widgets.Textarea(value=sample_model_answer.strip(), layout=box_layout)
student_answer_box = widgets.Textarea(value=sample_student_answer.strip(), layout=box_layout)

def labeled(title, widget):
    label = widgets.HTML(f"<b style='color:#white;'>{title}</b>")
    return widgets.VBox([label, widget])

form = widgets.VBox([
    labeled("❓ Question", question_box),
    labeled("✅ Model Answer", model_answer_box),
    labeled("✍️ Student Answer", student_answer_box),
])

evaluate_button = widgets.Button(
    description="Evaluate Answer",
    icon="check",
    button_style="success",
    layout=widgets.Layout(width="220px", height="45px", margin="15px 0")
)

output_area = widgets.Output()


def get_score_color(score):
    if score >= 7:
        return "#2ecc71"
    elif score >= 4:
        return "#f39c12"
    else:
        return "#e74c3c"


def on_evaluate_click(b):
    with output_area:
        clear_output()
        q, m, s = question_box.value, model_answer_box.value, student_answer_box.value

        if not q.strip() or not m.strip() or not s.strip():
            display(HTML("<p style='color:#white;'>⚠️ Please fill in all three fields.</p>"))
            return

        display(HTML("<p style='color:#666;'>⏳ Evaluating, please wait...</p>"))
        result = evaluate_answer(q, m, s)
        clear_output()

        # Graceful handling if something went wrong (API failure or unparseable response)
        if result["error"]:
            display(HTML(f"<p style='color:#white;'>⚠️ {result['error']}</p>"))
            return

        color = get_score_color(result["score"])
        remarks_html = "".join(f"<li style='margin-bottom:6px;'>{r}</li>" for r in result["remarks"])

        display(HTML(f"""
        <div style="border:1px solid #eee; border-radius:12px; padding:20px;
                    box-shadow:0 2px 8px rgba(0,0,0,0.08); font-family:sans-serif;">
            <div style="display:flex; align-items:center; margin-bottom:15px;">
                <div style="background:{color}; color:white; width:60px; height:60px;
                            border-radius:50%; display:flex; align-items:center; justify-content:center;
                            font-size:20px; font-weight:bold; margin-right:15px;">
                    {result['score']}/10
                </div>
                <h3 style="margin:0; color:white;">Evaluation Result</h3>
            </div>
            <b style="color:#white;">📝 Remarks:</b>
            <ul style="color:#white; padding-left:20px; margin-top:8px;">
                {remarks_html}
            </ul>
        </div>
        """))
        save_report(result)
        print("\n📁 Report saved to report.json")
evaluate_button.on_click(on_evaluate_click)
display(header, form, evaluate_button, output_area)

HTML(value='\n<div style="background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);\n            padding:…

Button(button_style='success', description='Evaluate Answer', icon='check', layout=Layout(height='45px', margi…

Output()